# Eclipse Data Processing (Figure 6)

This notebook creates deterministic datasets for eclipse evaluation:

1. `Data/eclipse_data/test_full_proteomics`
2. `Data/eclipse_data/test_significant_proteomics`
3. `Data/eclipse_data/train_full_proteomics_reduced`
4. `Data/eclipse_data/train_significant_proteomics_reduced`

Design:
- Test sets always use eclipse proteomics (`full` or `significant-only`).
- Non-proteomics modalities and labels come from the corresponding training-domain source.
- Reduced training sets remove eclipse sample IDs from the source dataset.
- We do **not** persist a separate extracted base test dump.


In [1]:
from pathlib import Path
import os
import sys
import pickle

import numpy as np
import pandas as pd

# Resolve repo root robustly
cwd = Path.cwd().resolve()
REPO_ROOT = None
for parent in [cwd] + list(cwd.parents):
    if (parent / "Data").exists() and (parent / "notebooks").exists():
        REPO_ROOT = parent
        break

# Fallback for terminals launched outside repo
if REPO_ROOT is None:
    candidate = Path('/mnt/jh/nas/PhD/Code/Ms_paint/code_repository')
    if candidate.exists():
        REPO_ROOT = candidate
    else:
        REPO_ROOT = cwd

os.chdir(REPO_ROOT)

# Ensure notebook-local modules are importable
for src in [REPO_ROOT / "notebooks", REPO_ROOT / "scripts"]:
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))

from data_processing.proteomics import prepare_proteomics_data, label_dict

print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: /mnt/nas/PhD/Code/Ms_paint/code_repository


In [2]:
# ---- Paths / constants ----
FASTA_PATH = Path("Data/Fasta_files/Ecoli.fasta")

RAW_MS_FILES = {
    "JH55": Path("Data/eclipse_data/MS_files/251208_JH55vsD.txt"),
    "JH6":  Path("Data/eclipse_data/MS_files/251208_JH6vsDH.txt"),
    "JH29": Path("Data/eclipse_data/MS_files/251208_JH29vsHC.txt"),
}

FULL_MS_DIR = Path("Data/eclipse_data/MS_files/all")
SIG_MS_DIR = Path("Data/eclipse_data/MS_files/statsFiltered_combined")
SIG_MS_DIR.mkdir(parents=True, exist_ok=True)

BASE_DIR = Path("Data/Base")
SIG_PROT_PATH = Path("Data/Base/X_proteomics_sig.pkl")

OUT_COMPOUND = Path("Data/eclipse_data/Compound_data")
OUT_COMPOUND_SIG = OUT_COMPOUND / "significant"
OUT_COMPOUND.mkdir(parents=True, exist_ok=True)
OUT_COMPOUND_SIG.mkdir(parents=True, exist_ok=True)

OUT_TEST_FULL = Path("Data/eclipse_data/test_full_proteomics")
OUT_TEST_SIG = Path("Data/eclipse_data/test_significant_proteomics")
OUT_TRAIN_FULL = Path("Data/eclipse_data/train_full_proteomics_reduced")
OUT_TRAIN_SIG = Path("Data/eclipse_data/train_significant_proteomics_reduced")

for p in [OUT_TEST_FULL, OUT_TEST_SIG, OUT_TRAIN_FULL, OUT_TRAIN_SIG]:
    p.mkdir(parents=True, exist_ok=True)

SPECIAL_COMPOUNDS = {"JH29", "JH55", "JH6"}
PVAL_THRESHOLD = 1.3  # -log10(p) threshold (~p<0.05)


In [3]:
# ---- Build significant-filtered eclipse MS file ----
def process_antibiotic_file(path: Path, ab_name: str, pval_threshold: float = 1.3) -> pd.DataFrame:
    df = pd.read_csv(path, sep="	")

    rep_cols = [c for c in df.columns if c.startswith(f"{ab_name}_")]
    if not rep_cols:
        raise ValueError(f"No replicate columns found for {ab_name} in {path}")

    pval_cols = [c for c in df.columns if "N: -Log Student's T-test p-value" in c]
    if len(pval_cols) != 1:
        raise ValueError(f"Expected 1 p-value column in {path}, found {pval_cols}")
    pval_col = pval_cols[0]

    sig_mask = (df[pval_col] > pval_threshold).fillna(False)
    df.loc[~sig_mask, rep_cols] = 0

    meta_cols = [c for c in df.columns if c.startswith("T: ")]
    key_col = "T: Protein.Ids"
    if key_col in meta_cols:
        meta_cols = [key_col] + [c for c in meta_cols if c != key_col]

    return df[rep_cols + meta_cols].copy()

processed = {ab: process_antibiotic_file(path, ab, PVAL_THRESHOLD) for ab, path in RAW_MS_FILES.items()}

key_col = "T: Protein.Ids"
for ab, df_ab in processed.items():
    if key_col not in df_ab.columns:
        raise KeyError(f"{key_col} missing in processed frame for {ab}")

ab_order = ["JH55", "JH6", "JH29"]
combined = processed[ab_order[0]].copy()
for ab in ab_order[1:]:
    rep_cols = [c for c in processed[ab].columns if not c.startswith("T: ")]
    combined = combined.merge(processed[ab][[key_col] + rep_cols], on=key_col, how="outer")

# Deterministic replicate ordering
intensity_cols = []
for ab in ab_order:
    cols = [c for c in combined.columns if c.startswith(f"{ab}_")]
    cols = sorted(cols, key=lambda x: int(x.split("_")[-1]))
    intensity_cols.extend(cols)

meta_cols = [c for c in combined.columns if c.startswith("T: ")]
if key_col in meta_cols:
    meta_cols = [key_col] + [c for c in meta_cols if c != key_col]

combined = combined[intensity_cols + meta_cols].copy()

# Normalize replicate names for prepare_proteomics_data():
# JH55_1 -> 55_1, JH6_1 -> 6_1, JH29_1 -> 29_1
rename_map = {}
for col in intensity_cols:
    if col.startswith("JH") and "_" in col:
        rename_map[col] = col[2:]
if rename_map:
    combined = combined.rename(columns=rename_map)
    intensity_cols = [rename_map.get(c, c) for c in intensity_cols]

sig_file = SIG_MS_DIR / "eclipse_data_statsFiltered.txt"
combined.to_csv(sig_file, sep="	", index=False)
print("Saved significant-filtered file:", sig_file)
print("shape:", combined.shape)


Saved significant-filtered file: Data/eclipse_data/MS_files/statsFiltered_combined/eclipse_data_statsFiltered.txt
shape: (2632, 20)


In [4]:
# ---- Build eclipse proteomics matrices (full + significant-only) ----
X_prot_full, _, y_full = prepare_proteomics_data(
    str(FASTA_PATH), str(FULL_MS_DIR), label_dict, exclude_labels=[]
)
X_prot_sig, _, y_sig = prepare_proteomics_data(
    str(FASTA_PATH), str(SIG_MS_DIR), label_dict, exclude_labels=[]
)

assert X_prot_full.shape[1] == 4401, f"Expected 4401 features (full), got {X_prot_full.shape[1]}"
assert X_prot_sig.shape[1] == 4401, f"Expected 4401 features (significant), got {X_prot_sig.shape[1]}"

X_prot_full.to_pickle(OUT_COMPOUND / "X_proteomics.pkl")
X_prot_sig.to_pickle(OUT_COMPOUND_SIG / "X_proteomics.pkl")

print("Saved full proteomics:", OUT_COMPOUND / "X_proteomics.pkl")
print("Saved significant proteomics:", OUT_COMPOUND_SIG / "X_proteomics.pkl")


[Warning] 42916 missing values filled with 0 in combined fingerprint matrix.


/mnt/nas/PhD/Code/Ms_paint/code_repository/notebooks/data_processing/proteomics.py:165: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(',', '.', regex=True)


[Warning] 20981 missing values filled with 0 in combined fingerprint matrix.


/mnt/nas/PhD/Code/Ms_paint/code_repository/notebooks/data_processing/proteomics.py:165: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(',', '.', regex=True)


Saved full proteomics: Data/eclipse_data/Compound_data/X_proteomics.pkl
Saved significant proteomics: Data/eclipse_data/Compound_data/significant/X_proteomics.pkl


In [5]:
# ---- Load source datasets (full + significant training domains) ----
def load_dataset(base_dir: Path, proteomics_override: Path | None = None):
    prot_path = proteomics_override if proteomics_override is not None else (base_dir / "X_proteomics.pkl")
    with open(prot_path, "rb") as f: Xp = pickle.load(f)
    with open(base_dir / "X_ecfp.pkl", "rb") as f: Xe = pickle.load(f)
    with open(base_dir / "X_cellpaint.pkl", "rb") as f: Xc = pickle.load(f)
    with open(base_dir / "X_mic.pkl", "rb") as f: Xm = pickle.load(f)
    with open(base_dir / "X_gc.pkl", "rb") as f: Xg = pickle.load(f)
    with open(base_dir / "y_labels.pkl", "rb") as f: y = pickle.load(f)

    if isinstance(y, dict):
        y = pd.Series(y)
    elif isinstance(y, pd.DataFrame):
        if "Label" in y.columns:
            y = y["Label"]
        elif y.shape[1] == 1:
            y = y.iloc[:, 0]
        else:
            raise ValueError(f"Unexpected y_labels columns in {base_dir}: {list(y.columns)}")

    for df in [Xp, Xe, Xc, Xm, Xg]:
        df.index = df.index.astype(str)
    y.index = y.index.astype(str)

    return {
        "X_proteomics": Xp,
        "X_ecfp": Xe,
        "X_cellpaint": Xc,
        "X_mic": Xm,
        "X_gc": Xg,
        "y_labels": y,
    }

full_src = load_dataset(BASE_DIR)
sig_src = load_dataset(BASE_DIR, proteomics_override=SIG_PROT_PATH)

X_prot_full.index = X_prot_full.index.astype(str)
X_prot_sig.index = X_prot_sig.index.astype(str)

test_ids = sorted(set(X_prot_full.index).intersection(X_prot_sig.index).intersection(full_src["X_proteomics"].index))
assert len(test_ids) > 0, "No overlapping test IDs found"

compounds = {sid.split("_")[0] for sid in test_ids}
assert compounds == SPECIAL_COMPOUNDS, f"Expected compounds {SPECIAL_COMPOUNDS}, got {compounds}"

print("test_ids:", test_ids)
print("n_test:", len(test_ids))


test_ids: ['JH29_1', 'JH29_2', 'JH29_3', 'JH29_4', 'JH55_1', 'JH55_2', 'JH55_3', 'JH55_4', 'JH6_1', 'JH6_2', 'JH6_3', 'JH6_4']
n_test: 12


In [6]:
# ---- Save two eclipse test sets ----
def save_test_set(out_dir: Path, X_prot_test: pd.DataFrame, src: dict, ids: list[str]):
    out_dir.mkdir(parents=True, exist_ok=True)
    X_prot_test.loc[ids].to_pickle(out_dir / "X_proteomics.pkl")
    src["X_ecfp"].loc[ids].to_pickle(out_dir / "X_ecfp.pkl")
    src["X_cellpaint"].loc[ids].to_pickle(out_dir / "X_cellpaint.pkl")
    src["X_mic"].loc[ids].to_pickle(out_dir / "X_mic.pkl")
    src["X_gc"].loc[ids].to_pickle(out_dir / "X_gc.pkl")
    src["y_labels"].loc[ids].to_pickle(out_dir / "y_labels.pkl")

save_test_set(OUT_TEST_FULL, X_prot_full, full_src, test_ids)
save_test_set(OUT_TEST_SIG, X_prot_sig, sig_src, test_ids)

print("Saved:", OUT_TEST_FULL)
print("Saved:", OUT_TEST_SIG)


Saved: Data/eclipse_data/test_full_proteomics
Saved: Data/eclipse_data/test_significant_proteomics


In [7]:
# ---- Save two reduced training sets (remove eclipse test IDs) ----
def save_reduced_train_set(out_dir: Path, src: dict, drop_ids: list[str]):
    out_dir.mkdir(parents=True, exist_ok=True)
    keep_ids = [sid for sid in src["X_proteomics"].index if sid not in set(drop_ids)]

    src["X_proteomics"].loc[keep_ids].to_pickle(out_dir / "X_proteomics.pkl")
    src["X_ecfp"].loc[keep_ids].to_pickle(out_dir / "X_ecfp.pkl")
    src["X_cellpaint"].loc[keep_ids].to_pickle(out_dir / "X_cellpaint.pkl")
    src["X_mic"].loc[keep_ids].to_pickle(out_dir / "X_mic.pkl")
    src["X_gc"].loc[keep_ids].to_pickle(out_dir / "X_gc.pkl")
    src["y_labels"].loc[keep_ids].to_pickle(out_dir / "y_labels.pkl")

    return len(keep_ids)

n_train_full = save_reduced_train_set(OUT_TRAIN_FULL, full_src, test_ids)
n_train_sig = save_reduced_train_set(OUT_TRAIN_SIG, sig_src, test_ids)

print("Saved:", OUT_TRAIN_FULL, "rows:", n_train_full)
print("Saved:", OUT_TRAIN_SIG, "rows:", n_train_sig)


Saved: Data/eclipse_data/train_full_proteomics_reduced rows: 188
Saved: Data/eclipse_data/train_significant_proteomics_reduced rows: 188


In [8]:
# ---- Validation ----
def load_saved(path: Path):
    with open(path / "X_proteomics.pkl", "rb") as f: Xp = pickle.load(f)
    with open(path / "X_ecfp.pkl", "rb") as f: Xe = pickle.load(f)
    with open(path / "X_cellpaint.pkl", "rb") as f: Xc = pickle.load(f)
    with open(path / "X_mic.pkl", "rb") as f: Xm = pickle.load(f)
    with open(path / "X_gc.pkl", "rb") as f: Xg = pickle.load(f)
    with open(path / "y_labels.pkl", "rb") as f: y = pickle.load(f)
    if isinstance(y, dict): y = pd.Series(y)
    elif isinstance(y, pd.DataFrame): y = y.iloc[:, 0]
    for d in [Xp, Xe, Xc, Xm, Xg]:
        d.index = d.index.astype(str)
    y.index = y.index.astype(str)
    return Xp, Xe, Xc, Xm, Xg, y

sets = {
    "test_full": load_saved(OUT_TEST_FULL),
    "test_sig": load_saved(OUT_TEST_SIG),
    "train_full": load_saved(OUT_TRAIN_FULL),
    "train_sig": load_saved(OUT_TRAIN_SIG),
}

for name, (Xp, Xe, Xc, Xm, Xg, y) in sets.items():
    idx = Xp.index
    assert idx.equals(Xe.index) and idx.equals(Xc.index) and idx.equals(Xm.index) and idx.equals(Xg.index) and idx.equals(y.index), f"Index mismatch in {name}"

assert len(set(sets["test_full"][0].index).intersection(set(sets["train_full"][0].index))) == 0
assert len(set(sets["test_sig"][0].index).intersection(set(sets["train_sig"][0].index))) == 0

print("Validation passed")
for name, ds in sets.items():
    print(f"{name}: {len(ds[0])} rows")


Validation passed
test_full: 12 rows
test_sig: 12 rows
train_full: 188 rows
train_sig: 188 rows
